# grep implementation: 
## Instructions

Search files for lines matching a search string and return all matching lines.

The Unix grep command searches files for lines that match a regular expression. Your task is to implement a simplified grep command, which supports searching for fixed strings.

The grep command takes three arguments:

1. The string to search for.
2. Zero or more flags for customizing the command's behavior.
3. One or more files to search in.


It then reads the contents of the specified files (in the order specified), finds the lines that contain the search string, and finally returns those lines in the order in which they were found. When searching in multiple files, each matching line is prepended by the file name and a colon (':').

Flags
The grep command supports the following flags:

- -n Prepend the line number and a colon (':') to each line in the output, placing the number after the filename (if present).
- -l Output only the names of the files that contain at least one matching line.
- -i Match using a case-insensitive comparison.
- -v Invert the program -- collect all lines that fail to match.
- -x Search only for lines where the search string matches the entire line.

## Step 1 — Search one pattern in one file

This is the minimum correct slice: one pattern, one file, stream line by line, print matching lines.

- re: 
  regex = re.compile(pattern)
  regex.search(line)

- sys.argv[1] first argument, sys.argv[2] second argument

In [ ]:
#!/usr/bin/env python3

import re
import sys


def search_file(pattern, filename):
    # Compile the pattern once before scanning the file.
    regex = re.compile(pattern)

    # Open the file in text mode.
    # errors="replace" prevents the program from crashing on bad characters.
    with open(filename, "r", encoding="utf-8", errors="replace") as f:
        # Read the file line by line instead of loading the whole file.
        for line in f:
            # If the regex matches this line, print the line.
            if regex.search(line):
                print(line, end="")


def main():
    # First CLI argument after the script name is the search pattern.
    pattern = sys.argv[1]

    # Second CLI argument after the script name is the filename.
    filename = sys.argv[2]

    # Search the file.
    search_file(pattern, filename)


if __name__ == "__main__":
    main()

## Step 2 — Add argparse

Now I replace raw sys.argv indexing with argparse, which gives better CLI behavior.
- parser = argparse.ArgumentParser(description:"")
- parser.add_argument("arg1")
- parser.add_argument("arg2")
- args = parser.parse_args()
- func(args.arg1, args.arg2)

In [ ]:
#!/usr/bin/env python3

import argparse
import re


def search_file(pattern, filename):
    # Compile the content regex once.
    regex = re.compile(pattern)

    # Stream the file line by line.
    with open(filename, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            if regex.search(line):
                print(line, end="")


def main():
    # Create a command-line argument parser.
    parser = argparse.ArgumentParser(
        description="Search for a regex pattern in a file."
    )

    # Add required positional argument for the content pattern.
    parser.add_argument("pattern")

    # Add required positional argument for the file path.
    parser.add_argument("file")

    # Parse arguments from the command line.
    args = parser.parse_args()

    # Run the search using parsed arguments.
    search_file(args.pattern, args.file)


if __name__ == "__main__":
    main()

# Step 3 — Handle invalid regex and unreadable files

Now I make the program more operationally robust.

Important SRE point: one bad input should produce a clear error. One unreadable file should not create an unclear traceback.

* for regex:
  try:
    with
  except re.error as exec

* for reading file:
  try:
        # try opening file
        with open(filename, mode='r', encoding="utf-8", errors='replace') as f:
  except SOError as err


* Exits the program using the return code from main().

In [ ]:
#!/usr/bin/env python3

import argparse
import re
import sys


def search_file(regex, filename):
    # Track whether this file had at least one match.
    found = False

    try:
        # Try opening the file.
        with open(filename, "r", encoding="utf-8", errors="replace") as f:
            # Stream file line by line.
            for line in f:
                # Check whether the compiled regex matches the line.
                if regex.search(line):
                    found = True
                    print(line, end="")

    except OSError as exc:
        # File-specific errors should be reported cleanly.
        print(f"warning: could not read {filename}: {exc}", file=sys.stderr)

    # Return whether this file had any match.
    return found


def main():
    parser = argparse.ArgumentParser(
        description="Search for a regex pattern in a file."
    )

    parser.add_argument("pattern")
    parser.add_argument("file")

    args = parser.parse_args()

    try:
        # Compile the regex once.
        regex = re.compile(args.pattern)

    except re.error as exc:
        # Invalid regex is a user-input error.
        print(f"error: invalid regex: {exc}", file=sys.stderr)
        return 2

    # Search the file and remember whether we found a match.
    found = search_file(regex, args.file)

    # Return Unix-style exit code.
    return 0 if found else 1


if __name__ == "__main__":
    raise SystemExit(main())

At this point, the tool has predictable exit behavior. That matters because CLI tools are often used in scripts, CI jobs, or operational workflows.

# Step 4 — Support multiple files and filename prefixes

Now the program accepts multiple files.

Requirement: if multiple files are searched, prefix matching lines with the filename.

In [ ]:
#!/usr/bin/env python3
import re
import sys
import argparse

def search_file(regex, filename, show_file):

    found = False

    try:
        with open(filename, mode='r', errors='replace', encoding='utf-8') as f:
            for line in f:
                if regex.search(line):
                    found = True
                    if show_file:
                        print(f"{filename}: {line}", end="")
                    else:
                        print(f"{line}", end="")

    except OSError as err:
        print(f"error in opeing the file {filename} is {err}")
    
    return found

def main():
    parser = argparse.ArgumentParser(
        description="search the pattern in file(s)"
    )
    parser.add_argument("pattern")
    parser.add_argument("files", nargs="+")
    args = parser.parse_args()

    try:
        regex = re.compile(args.pattern)
    except re.error as exc:
        print(f"error while making regex for pattern {args.pattern} is: {exec}")
        return 2
    
    show_filename = len(args.files)>1
    anymatch = False

    for filename in args.files:
        matched = search_file(regex, filename, show_filename)
        anymatch = anymatch or matched
    return 0 if anymatch else 1


if __name__ == "__main__":
    raise SystemExit(main())

At this point I’d say:

We are still streaming each file line by line. We are also handling OSError per file, so one unreadable file does not sink the whole scan.

# Step 5 — Add -n for line numbers

Now I add the first high-value flag: line numbers.

In [ ]:
#!/usr/bin/env python3

import argparse
import re
import sys


def search_file(regex, filename, show_filename, show_line_number):
    found = False

    try:
        with open(filename, "r", encoding="utf-8", errors="replace") as f:
            # enumerate gives us both line number and line content.
            # start=1 makes line numbers human-friendly.
            for line_number, line in enumerate(f, start=1):
                if regex.search(line):
                    found = True

                    # Build a prefix dynamically.
                    prefix_parts = []

                    # Add filename to prefix if needed.
                    if show_filename:
                        prefix_parts.append(filename)

                    # Add line number to prefix if requested.
                    if show_line_number:
                        prefix_parts.append(str(line_number))

                    # If we have a prefix, print prefix:line.
                    if prefix_parts:
                        print(":".join(prefix_parts) + ":" + line, end="")
                    else:
                        print(line, end="")

    except OSError as exc:
        print(f"warning: could not read {filename}: {exc}", file=sys.stderr)

    return found


def main():
    parser = argparse.ArgumentParser(
        description="Search for a regex pattern in one or more files."
    )

    parser.add_argument("pattern")
    parser.add_argument("files", nargs="+")

    # Optional flag to print line numbers.
    parser.add_argument(
        "-n",
        "--line-number",
        action="store_true",
        help="Show line numbers for matching lines.",
    )

    args = parser.parse_args()

    try:
        regex = re.compile(args.pattern)
    except re.error as exc:
        print(f"error: invalid regex: {exc}", file=sys.stderr)
        return 2

    show_filename = len(args.files) > 1
    any_match = False

    for filename in args.files:
        matched = search_file(
            regex=regex,
            filename=filename,
            show_filename=show_filename,
            show_line_number=args.line_number,
        )
        any_match = any_match or matched

    return 0 if any_match else 1


if __name__ == "__main__":
    raise SystemExit(main())

# Step 6 — Add -i for case-insensitive matching
Now I add case-insensitive matching.

- regex_flag = re.IGNORECASE if args.ignore_case else 0
- then pass this flag in regex.compile(args.pattern, regex_flag)
I’m still compiling once. The only difference is whether I pass re.IGNORECASE into re.compile.

In [ ]:
#!/usr/bin/env python3

import argparse
import re
import sys


def search_file(regex, filename, show_filename, show_line_number):
    found = False

    try:
        with open(filename, "r", encoding="utf-8", errors="replace") as f:
            for line_number, line in enumerate(f, start=1):
                if regex.search(line):
                    found = True

                    prefix_parts = []

                    if show_filename:
                        prefix_parts.append(filename)

                    if show_line_number:
                        prefix_parts.append(str(line_number))

                    if prefix_parts:
                        print(":".join(prefix_parts) + ":" + line, end="")
                    else:
                        print(line, end="")

    except OSError as exc:
        print(f"warning: could not read {filename}: {exc}", file=sys.stderr)

    return found


def main():
    parser = argparse.ArgumentParser(
        description="Search for a regex pattern in one or more files."
    )

    parser.add_argument("pattern")
    parser.add_argument("files", nargs="+")

    parser.add_argument(
        "-n",
        "--line-number",
        action="store_true",
        help="Show line numbers for matching lines.",
    )

    # Optional flag for case-insensitive matching.
    parser.add_argument(
        "-i",
        "--ignore-case",
        action="store_true",
        help="Perform case-insensitive matching.",
    )

    args = parser.parse_args()

    # If -i is passed, compile the regex with re.IGNORECASE.
    regex_flags = re.IGNORECASE if args.ignore_case else 0

    try:
        regex = re.compile(args.pattern, regex_flags)
    except re.error as exc:
        print(f"error: invalid regex: {exc}", file=sys.stderr)
        return 2

    show_filename = len(args.files) > 1
    any_match = False

    for filename in args.files:
        matched = search_file(
            regex=regex,
            filename=filename,
            show_filename=show_filename,
            show_line_number=args.line_number,
        )
        any_match = any_match or matched

    return 0 if any_match else 1


if __name__ == "__main__":
    raise SystemExit(main())

# Step 7 — Add -l for files with matches

Now I add -l, which prints only the filenames of files that contain at least one match.

In [ ]:
#!/usr/bin/env python3

import argparse
import re
import sys


def search_file(
    regex,
    filename,
    show_filename,
    show_line_number,
    files_with_matches,
):
    found = False

    try:
        with open(filename, "r", encoding="utf-8", errors="replace") as f:
            for line_number, line in enumerate(f, start=1):
                if regex.search(line):
                    found = True

                    # For -l, print filename once and stop scanning this file.
                    if files_with_matches:
                        print(filename)
                        return True

                    prefix_parts = []

                    if show_filename:
                        prefix_parts.append(filename)

                    if show_line_number:
                        prefix_parts.append(str(line_number))

                    if prefix_parts:
                        print(":".join(prefix_parts) + ":" + line, end="")
                    else:
                        print(line, end="")

    except OSError as exc:
        print(f"warning: could not read {filename}: {exc}", file=sys.stderr)

    return found


def main():
    parser = argparse.ArgumentParser(
        description="Search for a regex pattern in one or more files."
    )

    parser.add_argument("pattern")
    parser.add_argument("files", nargs="+")

    parser.add_argument("-n", "--line-number", action="store_true")
    parser.add_argument("-i", "--ignore-case", action="store_true")

    # Optional flag to only print file names that contain matches.
    parser.add_argument(
        "-l",
        "--files-with-matches",
        action="store_true",
        help="Print only names of files with at least one match.",
    )

    args = parser.parse_args()

    regex_flags = re.IGNORECASE if args.ignore_case else 0

    try:
        regex = re.compile(args.pattern, regex_flags)
    except re.error as exc:
        print(f"error: invalid regex: {exc}", file=sys.stderr)
        return 2

    show_filename = len(args.files) > 1
    any_match = False

    for filename in args.files:
        matched = search_file(
            regex=regex,
            filename=filename,
            show_filename=show_filename,
            show_line_number=args.line_number,
            files_with_matches=args.files_with_matches,
        )
        any_match = any_match or matched

    return 0 if any_match else 1


if __name__ == "__main__":
    raise SystemExit(main())

This is useful when I only care which logs contain a signal, not every matching line. It also short-circuits after the first match, which is efficient for large files.

# Step 8 — Support directories recursively